# Feature Engineering
1 . Target Variable : Rolling volatility
2

## Load the Data

In [14]:
import pandas as pd
import numpy as np

## Configuration

In [15]:
DATA_PATH = "../data/processed/cleaned_data.csv"

In [16]:
df = pd.read_csv(DATA_PATH, parse_dates=["date"])
df.head()

,open,high,low,close,volume,marketCap,crypto_name,date,log_return
0,-0.166334,-0.165950,-0.166175,-0.166323,-0.229533,-0.196627,Aave,2020-10-02,NaN
1,-0.166234,-0.153949,-0.166072,-0.156269,-0.229533,-0.196627,Aave,2020-10-03,53.253110
2,-0.156169,-0.155845,-0.156195,-0.156360,-0.229533,-0.196627,Aave,2020-10-04,-0.118554
3,-0.156265,-0.155837,-0.156373,-0.156256,-0.229533,-0.195439,Aave,2020-10-05,0.103119
4,-0.156147,-0.156154,-0.158155,-0.158323,-0.229472,-0.195681,Aave,2020-10-06,-2.627287


## 1 . Target Variable : Rolling volatility

In [17]:
df["rolling_volatility_14"] = (
    df.groupby("crypto_name")["log_return"]
    .transform(lambda x: x.rolling(window=14).std())
)

## 2. High- Low Range

In [18]:
df["high_low_range"] = df["high"] - df["low"]

## ATR (Average True Range -14 Days)
ATR measure market volatility using price movement
TR = Max [(H−L),∣H−Cp∣,∣L−Cp∣]

where 
H=Today’s high
L=Today’s low
Cp =Yesterday’s closing price
Max=Highest value of the three terms

In [19]:
def calculate_atr(group, window=14):
    high = group["high"]
    low = group["low"]
    close = group["close"]

    prev_close = close.shift(1)

    tr1 = high - low
    tr2 = (high - prev_close).abs()
    tr3 = (low - prev_close).abs()

    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    atr = tr.rolling(window=window).mean()
    return atr

# implement calculate_atr

In [20]:
df["ATR_14"] = df.groupby("crypto_name", group_keys=False).apply(calculate_atr)

## 6. Bollinger Brands
Bollinger Bands are a popular technical analysis tool developed by John Bollinger in the 1980s, consisting of a middle simple moving average (SMA) and two outer bands set at two standard deviations above and below it. They measure market volatility, expanding during high volatility and contracting during low, signaling potential overbought or oversold conditions. 

Middle Band: Typically a 20-period Simple Moving Average (SMA).

Upper Band: SMA + (20-period standard deviation *2).

Lower Band: SMA - (20-period standard deviation *2).

Volatility Indicator: Bands expand when volatility increases and contract during lower volatility, often preceding a "squeeze" or sharp movement. 

In [21]:
def bollinger_bands(group, window=20):
    ma = group["close"].rolling(window=window).mean()
    std = group["close"].rolling(window=window).std()

    upper = ma + 2 * std
    lower = ma - 2 * std

    width = upper - lower

    return width



In [22]:
df["bollinger_width_20"] = (
    df.groupby("crypto_name", group_keys=False)
    .apply(bollinger_bands)
)

## 7. Liquidity Features

In [23]:
df["liquidity_ratio"] = df["volume"] / df["marketCap"]
df["volume_pct_change"] = (
    df.groupby("crypto_name")["volume"]
    .transform(lambda x: x.pct_change())
)


## 8. Trend Features
Rolling Means

In [24]:
df["rolling_mean_7"] = (
    df.groupby("crypto_name")["close"]
    .transform(lambda x: x.rolling(7).mean())
)

df["rolling_mean_14"] = (
    df.groupby("crypto_name")["close"]
    .transform(lambda x: x.rolling(14).mean())
)


## 9. Momentum Feature

In [25]:
df["momentum_7"] = (
    df.groupby("crypto_name")["close"]
    .transform(lambda x: x - x.shift(7))
)


## 10. Drop NaN Rows

In [26]:
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

print("Final Shape After Feature Engineering:", df.shape)


Final Shape After Feature Engineering: (71900, 18)


## 11. Select Final Feature Set

In [27]:
features = [
    "open", "high", "low", "close",
    "volume", "marketCap",
    "high_low_range",
    "ATR_14",
    "bollinger_width_20",
    "liquidity_ratio",
    "volume_pct_change",
    "rolling_mean_7",
    "rolling_mean_14",
    "momentum_7"
]

target = "rolling_volatility_14"

X = df[features]
y = df[target]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)


Feature shape: (71900, 14)
Target shape: (71900,)


## 12. Save Engineered Dataset

In [28]:
df.to_csv("../data/processed/featured_data.csv", index=False)
